# Analise exploratória dos microdados do Enem 2023

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [2]:
file_path = '../data/raw/MICRODADOS_ENEM_2023.csv'
# [Acessar fonte](https://download.inep.gov.br/microdados/microdados_enem_2023.zip) 

df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1')
# aproximadamente 2m30s para rodar o dataset inteiro

# df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1', nrow=1000)

In [3]:
df_amostra.head(20);

## Especificações do dataset
- Estudar dimensionalidade do dataset
- Estudar preliminarmente a natureza das features
- Buscar por valores vazios

In [4]:
print(f"O dataset possui {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.\n")

O dataset possui 3933955 linhas e 76 colunas.



In [ ]:
df_amostra.dtypes.value_counts()

Visualizando os tipos de dados de cada variável:


NU_INSCRICAO       int64
NU_ANO             int64
TP_FAIXA_ETARIA    int64
TP_SEXO              str
TP_ESTADO_CIVIL    int64
                   ...  
Q021                 str
Q022                 str
Q023                 str
Q024                 str
Q025                 str
Length: 76, dtype: object

In [ ]:
df_amostra.info()

Visualizando especificações numéricas do dataset:


<bound method DataFrame.info of          NU_INSCRICAO  NU_ANO  TP_FAIXA_ETARIA TP_SEXO  TP_ESTADO_CIVIL  \
0        210059085136    2023               14       M                2   
1        210059527735    2023               12       M                2   
2        210061103945    2023                6       F                1   
3        210060214087    2023                2       F                1   
4        210059980948    2023                3       F                1   
...               ...     ...              ...     ...              ...   
3933950  210061959676    2023               12       M                1   
3933951  210061950911    2023                1       F                1   
3933952  210061965966    2023                3       F                1   
3933953  210061932304    2023                2       M                1   
3933954  210058924455    2023                3       F                1   

         TP_COR_RACA  TP_NACIONALIDADE  TP_ST_CONCLUSAO  TP_ANO_CON

In [ ]:
df_amostra.describe().T

### Construção do Dicionário de Dados

Será conduzido uma analise acerda das características de cada feature do dataset. 

O arquivo `resources/Dicionário_Microdados_Enem_2023.xlsx` representa o dicionário de dados oficial publicado juntamente ao dataset. A contrução do dicionário de dados neste nootebook será guiado por meio do dicionário oficial.

In [7]:
dados_dicionario = []

for coluna in df_amostra.columns:
    tipo = df_amostra[coluna].dtype
    valores_unicos = df_amostra[coluna].dropna().unique()
    
    # Pega até 5 exemplos de valores distintos
    exemplos = [str(v) for v in valores_unicos[:5]]
    
    dados_dicionario.append({
        'Nome da Variável': coluna,
        'Tipo de Dado': str(tipo),
        'Qtd Nulos': df_amostra[coluna].isnull().sum(),
        'Total Únicos': len(valores_unicos),
        'Exemplos Práticos': " | ".join(exemplos)
    })

df_dicionario = pd.DataFrame(dados_dicionario)

pd.set_option('display.max_rows', 150)
df_dicionario

,Nome da Variável,Tipo de Dado,Qtd Nulos,Total Únicos,Exemplos Práticos
0,NU_INSCRICAO,int64,0,3933955,210059085136 | 210059527735 | 210061103945 | 2...
1,NU_ANO,int64,0,1,2023
2,TP_FAIXA_ETARIA,int64,0,20,14 | 12 | 6 | 2 | 3
3,TP_SEXO,str,0,2,M | F
4,TP_ESTADO_CIVIL,int64,0,5,2 | 1 | 0 | 3 | 4
5,TP_COR_RACA,int64,0,6,1 | 3 | 2 | 0 | 5
6,TP_NACIONALIDADE,int64,0,5,1 | 0 | 4 | 2 | 3
7,TP_ST_CONCLUSAO,int64,0,4,1 | 2 | 3 | 4
8,TP_ANO_CONCLUIU,int64,0,18,17 | 16 | 0 | 12 | 1
9,TP_ESCOLA,int64,0,3,1 | 2 | 3


Após comparar o dicionário acima com o dicionário oficial, não foi encontrado divergências.

### Classificação das features

Para facilitar as manipulações futuras, podemos separar as features em 4 grandes grupos, baseando-se em suas descrições no dicionário oficial:

1. **Dados Categóricos (Qualitativos):** 
   - São features que representam características, grupos ou códigos. 
   - Exemplo: `TP_FAIXA_ETARIA`.
   
2. **Dados Numéricos (Quantitativos Contínuos):**
   - São os valores que representam medidas ou grandezas contínuas.
   - Exemplo: `NU_NOTA_REDACAO`.

3. **Dados Textuais ("Escritos" ou Strings Literais):**
   - Dados que representam texto descritivo.
   - Exemplo: `NO_MUNICIPIO`, `TX_GABARITO`.
   
4. **Identificadores (Chave):**
   - Servem apenas como indexador ou identificador.
   - Exemplo: `NU_INSCRICAO`.

In [10]:
# Indentificadores 
colunas_identificadoras = ['NU_INSCRICAO', 'CO_MUNICIPIO_ESC', 'CO_MUNICIPIO_PROVA'] 

# Dados Numéricos 
colunas_numericas = [
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 
    'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 
    'NU_NOTA_COMP5', 'NU_NOTA_REDACAO'
]

# Dados Escritos / Textuais Liberais
colunas_textuais = [
    'NO_MUNICIPIO_ESC', 'SG_UF_ESC',
    'NO_MUNICIPIO_PROVA', 'SG_UF_PROVA',
    'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT',
    'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT'
]

# Dados Categóricos (o restante)
colunas_categoricas = [
    col for col in df_amostra.columns 
    if col not in colunas_numericas and col not in colunas_textuais and col not in colunas_identificadoras
]

In [ ]:
dicionario_categorico = {}

for coluna in colunas_categoricas:
    valores_unicos = df_amostra[coluna].dropna().unique()
    contagem = df_amostra[coluna].value_counts().sort_index()
    
    dicionario_categorico[coluna] = {
        'Qtd_Valores_Unicos': len(valores_unicos),
        'Qtd_Nulos': df_amostra[coluna].isnull().sum(),
        'Valores': contagem.to_dict()
    }

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Criar um DataFrame resumido para visualização
resumo_categorico = []
for coluna, dados in dicionario_categorico.items():
    resumo_categorico.append({
        'Coluna': coluna,
        'Qtd_Valores_Únicos': dados['Qtd_Valores_Unicos'],
        'Qtd_Nulos': dados['Qtd_Nulos'],
        'Valores': str(dados['Valores'])
    })

df_resumo = pd.DataFrame(resumo_categorico)
display(df_resumo)

,Coluna,Qtd_Valores_Únicos,Qtd_Nulos,Valores
0,NU_ANO,1,0,{2023: 3933955}
1,TP_FAIXA_ETARIA,20,0,"{1: 347434, 2: 753800, 3: 905047, 4: 431592, 5: 267383, 6: 183401, 7: 137884, 8: 111813, 9: 91359, 10: 73127, 11: 246292, 12: 133381, 13: 96927, 14: 67135, 15: 40791, 16: 24619, 17: 13439, 18: 5503, 19: 2161, 20: 867}"
2,TP_SEXO,2,0,"{'F': 2411185, 'M': 1522770}"
3,TP_ESTADO_CIVIL,5,0,"{0: 171900, 1: 3491857, 2: 200456, 3: 64933, 4: 4809}"
4,TP_COR_RACA,6,0,"{0: 52575, 1: 1575848, 2: 509511, 3: 1706798, 4: 64512, 5: 24711}"
5,TP_NACIONALIDADE,5,0,"{0: 2153, 1: 3842681, 2: 73429, 3: 7112, 4: 8580}"
6,TP_ST_CONCLUSAO,4,0,"{1: 1895301, 2: 1401164, 3: 620067, 4: 17423}"
7,TP_ANO_CONCLUIU,18,0,"{0: 2243134, 1: 418530, 2: 264183, 3: 149798, 4: 136449, 5: 104195, 6: 85411, 7: 65549, 8: 54769, 9: 46654, 10: 39920, 11: 35512, 12: 29001, 13: 27421, 14: 24639, 15: 20998, 16: 20471, 17: 167321}"
8,TP_ESCOLA,3,0,"{1: 2532796, 2: 1166540, 3: 234619}"
9,TP_ENSINO,2,2594874,"{1.0: 1332195, 2.0: 6886}"


In [ ]:
n_cols = 2
n_rows = math.ceil(len(colunas_categoricas) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(colunas_categoricas):
    sns.countplot(
        data=df_amostra,
        x=col,
        order=df_amostra[col].value_counts().index,
        ax=axes[i]
    )
    
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)

# remove eixos vazios
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()